# VIIRS and GOES-R Case Viewer

This notebook visualizes one wildfire case to inspect how data flows before modeling:

- VIIRS/FirePred raw image sequence over one `pred` input window.
- GOES-R FDCF `mask_fixed` and `frp_fixed` rasters over the same dates/period.
- Fire overlays where available: VIIRS active/burned bands and GOES active-fire mask codes.

Run this on the HPC where the raw raster paths are available.

## 1. Configuration

Set `FIRE_ID` to a known 2021 test fire or leave it as `None` to auto-select the first case with both VIIRS and GOES data.

`WINDOW_START` is the start index in the sorted VIIRS_Day file list. For `pred`, the observed input window is `TS_LENGTH` VIIRS observations and the next VIIRS file is the target timestamp.

In [ ]:
from pathlib import Path

RAW_VIIRS_ROOT = Path('/home/jlc3q/data/SatFire/ts-satfire')
GOES_ROOT = Path('/home/jlc3q/data/GOES_clipped_tif_common_wgs84')
PRED_DATASET_ROOT = Path('/home/jlc3q/data/SatFire/dataset/pred')

# Use None for automatic selection. Good known examples from previous test logs are listed below.
FIRE_ID = None
PREFERRED_FIRE_IDS = [
    'US_2021_CA3604711863120210910',
    'US_2021_CA3568711855020210818',
    'US_2021_WA4828511853120210713',
    'US_2021_AZ3368910927620210616',
]

TS_LENGTH = 4
WINDOW_START = 0
CROP_SIZE = 256
CROP_OFFSET = 128
MAX_GOES_FRAMES_PER_DAY = 6
MAX_GOES_TOTAL_FRAMES = 36

MASK_DIR_NAMES = ('mask_fixed', 'mask')
FRP_DIR_NAMES = ('frp_fixed', 'frp')
FIRE_MASK_CODES = set(range(10, 16)) | set(range(20, 26)) | set(range(30, 36))
TIFF_SUFFIXES = {'.tif', '.tiff'}

## 2. Utilities

In [ ]:
import math
import os
import re
from collections import defaultdict
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import rasterio


def parse_timestamp_from_name(name):
    stem = Path(name).stem
    patterns = [
        (r'(?<!\d)(20\d{6}T\d{6})(?!\d)', '%Y%m%dT%H%M%S'),
        (r'(?<!\d)(20\d{12})(?!\d)', '%Y%m%d%H%M%S'),
        (r'(?<!\d)(20\d{8})(?!\d)', '%Y%m%d%H'),
        (r'(?<!\d)(20\d{6})(?!\d)', '%Y%m%d'),
        (r'(?<!\d)s(20\d{2})(\d{3})(\d{6})(?!\d)', None),
    ]
    for pattern, fmt in patterns:
        match = re.search(pattern, stem)
        if not match:
            continue
        if fmt is not None:
            try:
                return datetime.strptime(match.group(1), fmt)
            except ValueError:
                continue
        year, doy, hms = match.groups()
        try:
            return datetime.strptime(f'{year}{doy}{hms}', '%Y%j%H%M%S')
        except ValueError:
            continue
    try:
        return datetime.strptime(stem.replace('_VIIRS_Day', ''), '%Y-%m-%d')
    except ValueError:
        return None


def read_raster(path):
    with rasterio.open(path) as src:
        arr = src.read()
        profile = src.profile
    return np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0), profile


def crop_center_256(arr, offset=CROP_OFFSET, size=CROP_SIZE):
    if arr.ndim == 2:
        return arr[offset:offset+size, offset:offset+size]
    return arr[:, offset:offset+size, offset:offset+size]


def robust_norm(x, p_low=2, p_high=98):
    x = np.asarray(x, dtype=np.float32)
    finite = np.isfinite(x)
    if not finite.any():
        return np.zeros_like(x, dtype=np.float32)
    lo, hi = np.nanpercentile(x[finite], [p_low, p_high])
    if hi <= lo:
        return np.zeros_like(x, dtype=np.float32)
    return np.clip((x - lo) / (hi - lo), 0, 1)


def find_event_dir(goes_root, event_id):
    direct = goes_root / event_id
    if direct.is_dir():
        return direct
    for year_dir in sorted(path for path in goes_root.iterdir() if path.is_dir()):
        candidate = year_dir / event_id
        if candidate.is_dir():
            return candidate
    matches = [path for path in goes_root.rglob(event_id) if path.is_dir()]
    return matches[0] if matches else None


def first_existing_subdir(event_dir, names):
    for name in names:
        subdir = event_dir / name
        if subdir.is_dir():
            return subdir
    return None


def raster_files(folder):
    if folder is None or not folder.is_dir():
        return []
    return sorted(path for path in folder.iterdir() if path.suffix.lower() in TIFF_SUFFIXES)


def viirs_files_for_fire(fire_id):
    folder = RAW_VIIRS_ROOT / fire_id / 'VIIRS_Day'
    return sorted(path for path in folder.glob('*.tif')) if folder.is_dir() else []


def choose_fire_id():
    candidates = []
    if FIRE_ID:
        candidates.append(FIRE_ID)
    candidates.extend(PREFERRED_FIRE_IDS)
    test_files = sorted((PRED_DATASET_ROOT / 'dataset_test').glob(f'pred_*_img_seqtoseql_{TS_LENGTH}i_1.npy'))
    for path in test_files:
        candidates.append(path.name.replace('pred_', '').replace(f'_img_seqtoseql_{TS_LENGTH}i_1.npy', ''))
    seen = set()
    for fire_id in candidates:
        if fire_id in seen:
            continue
        seen.add(fire_id)
        viirs = viirs_files_for_fire(fire_id)
        event_dir = find_event_dir(GOES_ROOT, fire_id)
        if viirs and event_dir is not None:
            return fire_id
    raise RuntimeError('No fire with both VIIRS_Day and GOES event directory was found. Set FIRE_ID manually.')

## 3. Select One Case and VIIRS Window

In [ ]:
fire_id = choose_fire_id()
viirs_files = viirs_files_for_fire(fire_id)
event_dir = find_event_dir(GOES_ROOT, fire_id)
mask_dir = first_existing_subdir(event_dir, MASK_DIR_NAMES)
frp_dir = first_existing_subdir(event_dir, FRP_DIR_NAMES)
mask_files = raster_files(mask_dir)
frp_files = raster_files(frp_dir)

if WINDOW_START + TS_LENGTH >= len(viirs_files):
    raise ValueError(f'WINDOW_START={WINDOW_START}, TS_LENGTH={TS_LENGTH} exceeds VIIRS file count={len(viirs_files)}')

window_viirs_files = viirs_files[WINDOW_START:WINDOW_START + TS_LENGTH]
target_viirs_file = viirs_files[WINDOW_START + TS_LENGTH]
window_dates = [parse_timestamp_from_name(path.name).date().isoformat() for path in window_viirs_files]
target_date = parse_timestamp_from_name(target_viirs_file.name).date().isoformat()

print('fire_id:', fire_id)
print('event_dir:', event_dir)
print('VIIRS file count:', len(viirs_files))
print('GOES mask files:', len(mask_files), 'dir:', mask_dir)
print('GOES FRP files:', len(frp_files), 'dir:', frp_dir)
print('observed VIIRS dates:', window_dates)
print('target next VIIRS date:', target_date)

## 4. VIIRS Sequence

For each VIIRS observation, the notebook shows a thermal-like grayscale band and overlays available fire labels:

- Red: active fire band from `VIIRS_Day` band 7 (`array_day[6] > 0`) if present.
- Orange: burned area / label band from `VIIRS_Day` band 8 (`array_day[7] > 0`) if present.

Band indices follow the TS-SatFire generator convention. The display is for visual inspection, not scientific radiance calibration.

In [ ]:
def plot_viirs_sequence(paths):
    n = len(paths)
    fig, axes = plt.subplots(1, n, figsize=(4*n, 4), squeeze=False)
    for ax, path in zip(axes.ravel(), paths):
        arr, _ = read_raster(path)
        crop = crop_center_256(arr)
        # TS-SatFire uses band index 3/I4-like thermal signal heavily for visualization.
        base_idx = 3 if crop.shape[0] > 3 else 0
        base = robust_norm(crop[base_idx])
        ax.imshow(base, cmap='gray')
        title = parse_timestamp_from_name(path.name).date().isoformat()
        if crop.shape[0] > 6:
            af = crop[6] > 0
            ax.imshow(np.where(af, 1.0, np.nan), cmap='Reds', alpha=0.55, vmin=0, vmax=1)
            title += f'\nAF px={int(af.sum())}'
        if crop.shape[0] > 7:
            ba = crop[7] > 0
            ax.imshow(np.where(ba, 1.0, np.nan), cmap='autumn', alpha=0.35, vmin=0, vmax=1)
            title += f' BA px={int(ba.sum())}'
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()

plot_viirs_sequence(window_viirs_files)

## 5. GOES Files Over the Same VIIRS Dates

This groups GOES FDCF files by date and counts how many `mask_fixed` and `frp_fixed` observations exist for each VIIRS observation day.

In [ ]:
def group_by_date(paths):
    grouped = defaultdict(list)
    for path in paths:
        ts = parse_timestamp_from_name(path.name)
        if ts is not None:
            grouped[ts.date().isoformat()].append((ts, path))
    return grouped

mask_by_date = group_by_date(mask_files)
frp_by_date = group_by_date(frp_files)

for day in window_dates + [target_date]:
    print(day, 'mask_count=', len(mask_by_date.get(day, [])), 'frp_count=', len(frp_by_date.get(day, [])))

## 6. GOES Mask Sequence for the VIIRS Window Dates

GOES FDCF mask rasters are lower spatial resolution than VIIRS but clipped to the same fire extent. Active-fire pixels are displayed from FDCF fire mask code ranges.

In [ ]:
def sample_evenly(items, max_items):
    if len(items) <= max_items:
        return items
    idx = np.linspace(0, len(items) - 1, max_items).round().astype(int)
    return [items[i] for i in idx]


def collect_window_goes(grouped, dates, max_per_day=MAX_GOES_FRAMES_PER_DAY, max_total=MAX_GOES_TOTAL_FRAMES):
    selected = []
    for day in dates:
        selected.extend(sample_evenly(grouped.get(day, []), max_per_day))
    if len(selected) > max_total:
        selected = sample_evenly(selected, max_total)
    return selected


def plot_goes_mask_sequence(items):
    if not items:
        print('No GOES mask frames found for selected VIIRS dates.')
        return
    n = len(items)
    cols = min(6, n)
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(3.2*cols, 3.2*rows), squeeze=False)
    for ax in axes.ravel():
        ax.axis('off')
    for ax, (ts, path) in zip(axes.ravel(), items):
        arr, _ = read_raster(path)
        m = arr[0]
        active = np.isin(m, list(FIRE_MASK_CODES))
        ax.imshow(active, cmap='Reds', vmin=0, vmax=1)
        ax.set_title(f'{ts:%m-%d %H:%M}\nactive={int(active.sum())}')
        ax.axis('off')
    plt.tight_layout()

mask_window_items = collect_window_goes(mask_by_date, window_dates)
print('selected GOES mask frames:', len(mask_window_items))
plot_goes_mask_sequence(mask_window_items)

## 7. GOES FRP Sequence for the VIIRS Window Dates

FRP is shown with a robust percentile stretch. If active fire is detected, bright spots should appear over the clipped fire extent.

In [ ]:
def plot_goes_frp_sequence(items):
    if not items:
        print('No GOES FRP frames found for selected VIIRS dates.')
        return
    n = len(items)
    cols = min(6, n)
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(3.2*cols, 3.2*rows), squeeze=False)
    for ax in axes.ravel():
        ax.axis('off')
    for ax, (ts, path) in zip(axes.ravel(), items):
        arr, _ = read_raster(path)
        frp = np.maximum(arr[0], 0)
        ax.imshow(robust_norm(frp), cmap='inferno')
        ax.set_title(f'{ts:%m-%d %H:%M}\nmax={float(frp.max()):.1f}')
        ax.axis('off')
    plt.tight_layout()

frp_window_items = collect_window_goes(frp_by_date, window_dates)
print('selected GOES FRP frames:', len(frp_window_items))
plot_goes_frp_sequence(frp_window_items)

## 8. GOES Sub-Daily Temporal Summary for the Same Dates

This cell recreates the current model's `[T, 96, 4]` style features directly from the raw GOES rasters for the selected VIIRS dates:

1. `active_pixel_count`
2. `mask_coverage`
3. `frp_mean`
4. `frp_max`

This is useful for checking whether the GOES branch is seeing meaningful temporal variation even when the raw maps are coarse.

In [ ]:
def mask_bin_features(paths):
    if not paths:
        return 0.0, 0.0
    active_counts, coverages = [], []
    for path in paths:
        arr, _ = read_raster(path)
        fire_mask = np.isin(arr[0], list(FIRE_MASK_CODES))
        active_counts.append(float(fire_mask.sum()))
        coverages.append(float(fire_mask.mean()))
    return float(max(active_counts)), float(max(coverages))


def frp_bin_features(paths):
    if not paths:
        return 0.0, 0.0
    means, maxima = [], []
    for path in paths:
        arr, _ = read_raster(path)
        positive = arr[0][arr[0] > 0]
        if positive.size == 0:
            means.append(0.0)
            maxima.append(0.0)
        else:
            means.append(float(positive.mean()))
            maxima.append(float(positive.max()))
    return float(np.mean(means)), float(max(maxima))


def build_bin_store(paths, bin_minutes=15):
    store = defaultdict(lambda: defaultdict(list))
    for path in paths:
        ts = parse_timestamp_from_name(path.name)
        if ts is None:
            continue
        bin_idx = (ts.hour * 60 + ts.minute) // bin_minutes
        store[ts.date().isoformat()][bin_idx].append(path)
    return store


def build_goes_feature_matrix(dates, bin_minutes=15):
    bins_per_day = math.ceil(24 * 60 / bin_minutes)
    mask_store = build_bin_store(mask_files, bin_minutes=bin_minutes)
    frp_store = build_bin_store(frp_files, bin_minutes=bin_minutes)
    out = np.zeros((len(dates), bins_per_day, 4), dtype=np.float32)
    for t, day in enumerate(dates):
        for b in range(bins_per_day):
            active_count, mask_cov = mask_bin_features(mask_store.get(day, {}).get(b, []))
            frp_mean, frp_max = frp_bin_features(frp_store.get(day, {}).get(b, []))
            out[t, b] = [active_count, mask_cov, frp_mean, frp_max]
    return out

features = build_goes_feature_matrix(window_dates, bin_minutes=15)
feature_names = ['active_pixel_count', 'mask_coverage', 'frp_mean', 'frp_max']

fig, axes = plt.subplots(len(feature_names), 1, figsize=(16, 8), sharex=True)
for i, (ax, name) in enumerate(zip(axes, feature_names)):
    im = ax.imshow(features[:, :, i], aspect='auto', cmap='viridis')
    ax.set_ylabel(name)
    ax.set_yticks(range(len(window_dates)))
    ax.set_yticklabels(window_dates)
    fig.colorbar(im, ax=ax, fraction=0.02, pad=0.01)
axes[-1].set_xlabel('15-minute bin of day')
plt.tight_layout()

print('feature tensor shape:', features.shape, '[T, bins, F]')
print('nonzero bins by feature:', {name: int((features[:, :, i] > 0).sum()) for i, name in enumerate(feature_names)})

## 9. Interpretation Checklist

Use the figures above to answer:

- Does VIIRS show active/burned fire pixels in the selected window?
- Does GOES FDCF mask/FRP show activity on the same days?
- Are GOES detections temporally clustered before/after VIIRS observations?
- Is GOES mostly empty for this fire? If yes, this case may not help the GOES branch.
- Does GOES show movement/direction visually? If not, scalar summaries may be insufficient and spatial descriptors or cross-attention may be needed.